In [1]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("=== 05a: Environment Check ===")
print("Python:", sys.version.split()[0])
print("Drive mounted:", Path("/content/drive/MyDrive").is_dir())

for package in (
    "torch", "torchvision", "ultralytics",
    "numpy", "pycocotools", "opencv-python", "Pillow"
):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch import: NOT AVAILABLE")

print("No packages installed. No files modified. No inference performed.")

=== 05a: Environment Check ===
Python: 3.13.18
Drive mounted: False
torch: 2.14.1+cpu
torchvision: 0.29.1+cpu
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11
opencv-python: 5.0.0.93
Pillow: 11.3.0
CUDA available: False
No packages installed. No files modified. No inference performed.


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import hashlib
import json
import subprocess

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = project / "code"

checks = {
    "Checkpoint": (
        project / "models/yolov8s.pt",
        "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a",
    ),
    "Split manifest": (
        repo / "experiments/notebook05/coco2017_split_manifest_v1.json",
        "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee",
    ),
    "Quality gate": (
        repo / "experiments/notebook05/shared_INT8_clean_quality_gate_v1.json",
        "dfc04556e731df68af9375c161778e41b87f66601856265703ea8acd69fb64f3",
    ),
}

for label, (path, expected_sha) in checks.items():
    actual_sha = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual_sha != expected_sha:
        raise RuntimeError(f"Integrity mismatch: {label}")
    print(f"{label}: PASS")

split = json.loads(checks["Split manifest"][0].read_bytes())
quality_gate = json.loads(checks["Quality gate"][0].read_bytes())

development_ids = split["image_ids"]["development"]
held_out_ids = split["image_ids"]["held_out"]

assert len(development_ids) == len(set(development_ids)) == 1000
assert len(held_out_ids) == len(set(held_out_ids)) == 4000
assert not set(development_ids) & set(held_out_ids)

annotation_path = (
    project / "datasets/coco2017/annotations/instances_val2017.json"
)
if hashlib.sha256(annotation_path.read_bytes()).hexdigest() != (
    split["annotation_sha256"]
):
    raise RuntimeError("Annotation integrity mismatch.")

head = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=repo, capture_output=True, text=True, check=True,
).stdout.strip()

print("Annotations: PASS")
print("Development images:", len(development_ids))
print("Held-out images:", len(held_out_ids))
print("Git HEAD:", head)
print("Quality criteria:", quality_gate["criteria"])
print("No files modified. No model loaded or inference performed.")

Checkpoint: PASS
Split manifest: PASS
Quality gate: PASS
Annotations: PASS
Development images: 1000
Held-out images: 4000
Git HEAD: 24adf45fc73d8ccc16f356bd072260d925c46b59
Quality criteria: {'candidate_minus_FP32_AP50_95_minimum': -0.01, 'candidate_minus_FP32_AR100_minimum': -0.01, 'both_required': True, 'finite_predictions_and_complete_image_execution_required': True}
No files modified. No model loaded or inference performed.


In [4]:
import subprocess
import sys
from importlib.metadata import version

packages = [
    "ultralytics==8.4.189",
    "nvidia-ml-py==13.615.71",
    "ultralytics-thop==2.2.5",
    "ultralytics-platform==0.1.108",
    "jedi==0.20.1",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", *packages],
    check=True,
)

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)
print("--- Dependency check ---")
print(check.stdout.strip() or check.stderr.strip() or "(no output)")

if check.returncode != 0:
    raise RuntimeError("Dependency check failed. Stop before model loading.")

from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes
from pycocotools.cocoeval import COCOeval

print("\nRequired imports: PASS")
print("Ultralytics:", version("ultralytics"))
print("PyTorch:", version("torch"))
print("torchvision:", version("torchvision"))
print("No model loaded. No inference performed.")

--- Dependency check ---
No broken requirements found.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

Required imports: PASS
Ultralytics: 8.4.189
PyTorch: 2.14.1+cpu
torchvision: 0.29.1+cpu
No model loaded. No inference performed.


In [5]:
from google.colab import files
from pathlib import Path
import hashlib
import io
import subprocess
import sys
import zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Select exactly one ZIP file.")

content = next(iter(uploaded.values()))
if hashlib.sha256(content).hexdigest() != (
    "e7414f8e09fb51e072019cae3c5dceae212f445dbbfe95eee6d5cbd16675246b"
):
    raise RuntimeError("Package SHA256 mismatch.")

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
expected_paths = {
    "src/baseline05a_runner_v1.py",
    "tests/test_baseline05a_runner_v1.py",
    "docs/research_reports/05a_runner_notes_v1.md",
}

with zipfile.ZipFile(io.BytesIO(content)) as archive:
    if set(archive.namelist()) != expected_paths:
        raise RuntimeError("Unexpected package contents.")

    payloads = {name: archive.read(name) for name in expected_paths}

    for name, data in payloads.items():
        destination = repo / name
        if destination.exists() and destination.read_bytes() != data:
            raise RuntimeError(f"Different file already exists: {name}")

    for name, data in payloads.items():
        destination = repo / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            with destination.open("xb") as stream:
                stream.write(data)
        print("Verified:", name)

subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", str(repo / "tests"),
        "-p", "test_baseline05a_runner_v1.py", "-v",
    ],
    check=True,
)

print("Package installed and storage/provenance tests passed.")
print("No model loaded. No inference, commit or push performed.")

Saving baseline05a_runner_v1.zip to baseline05a_runner_v1.zip
Verified: src/baseline05a_runner_v1.py
Verified: docs/research_reports/05a_runner_notes_v1.md
Verified: tests/test_baseline05a_runner_v1.py
Package installed and storage/provenance tests passed.
No model loaded. No inference, commit or push performed.


In [6]:
from pathlib import Path
import hashlib
import sys

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
module_path = repo / "src/baseline05a_runner_v1.py"

if hashlib.sha256(module_path.read_bytes()).hexdigest() != (
    "f1069948c16a8899d422930b6dfb2c4ce0f900e5e27e358e0761d588b11492b3"
):
    raise RuntimeError("Runner source differs from the reviewed package.")

src = str(repo / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from baseline05a_runner_v1 import prepare

session = prepare(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance"
)
smoke_result = session.smoke()

Session prepared on CPU; two synthetic warm-up forwards completed.
No development image evaluated yet by preparation.
Fingerprint: 5fab9260071987f89edf01099cd63925ca4743fb47f925063e9fa17aaac59c39
Smoke images: [1425, 1490]
Prediction counts: {'fp32_clean': 99, 'int8_channel_clean': 97}
State integrity and finite-output checks: PASS


In [7]:
from pathlib import Path
import hashlib

# Complete or resume all paired development-image chunks.
status = session.run(max_new_chunks=None)

if status["completed_images_per_model"] != 1000:
    raise RuntimeError("Evaluation requires 1000 completed images per model.")

# Evaluate both models and save the metrics and quality-gate decision.
report = session.evaluate()

evaluation_path = session.out / "evaluation_and_quality_gate.json.gz"
evaluation_sha = hashlib.sha256(evaluation_path.read_bytes()).hexdigest()

fp32 = report["metrics"]["fp32_clean"]
int8 = report["metrics"]["int8_channel_clean"]

card = f"""# Shared INT8 clean baseline — development validation

Status: {report["quality_gate_decision"]}
Images: 1000 frozen development images
Session fingerprint: {session.fingerprint}
Evaluation SHA256: {evaluation_sha}

| Metric | FP32 | Logical INT8 per channel | INT8 minus FP32 |
| --- | ---: | ---: | ---: |
| AP50:95 | {fp32["AP50_95"]:.9f} | {int8["AP50_95"]:.9f} | {int8["AP50_95"] - fp32["AP50_95"]:+.9f} |
| AP50 | {fp32["AP50"]:.9f} | {int8["AP50"]:.9f} | {int8["AP50"] - fp32["AP50"]:+.9f} |
| AP75 | {fp32["AP75"]:.9f} | {int8["AP75"]:.9f} | {int8["AP75"] - fp32["AP75"]:+.9f} |
| AR100 | {fp32["AR100"]:.9f} | {int8["AR100"]:.9f} | {int8["AR100"] - fp32["AR100"]:+.9f} |

## Interpretation

The recorded provisional engineering gate allows at most 0.01 AP50:95
loss and 0.01 AR100 loss. Both conditions are required.
This is not a statistical equivalence test or final held-out evaluation.

Weights use logical INT8 with per-output-channel scales; computation
and activations remain FP32. No faults were injected.
No native INT8 hardware performance claim is made.

Detailed results and paired prediction chunks:
results/notebook05a/shared_clean_cpu_v1/

No commit or push is performed by this execution.
"""

card_path = (
    session.repo / "docs/research_reports/"
    "05a_shared_INT8_clean_baseline_card_v1.md"
)
card_bytes = card.encode("utf-8")
card_path.parent.mkdir(parents=True, exist_ok=True)

if card_path.exists():
    if card_path.read_bytes() != card_bytes:
        raise RuntimeError("A different baseline card exists; preserved.")
else:
    with card_path.open("xb") as stream:
        stream.write(card_bytes)

print("\n=== Automatic validation completed ===")
print("Quality gate:", report["quality_gate_decision"])
print("Evaluation:", evaluation_path)
print("Evaluation SHA256:", evaluation_sha)
print("Baseline card:", card_path)
print("No fault injection. No held-out evaluation. No commit or push.")

Saved paired chunk 1/40 (25 images)
Saved paired chunk 2/40 (25 images)
Saved paired chunk 3/40 (25 images)
Saved paired chunk 4/40 (25 images)
Saved paired chunk 5/40 (25 images)
Saved paired chunk 6/40 (25 images)
Saved paired chunk 7/40 (25 images)
Saved paired chunk 8/40 (25 images)
Saved paired chunk 9/40 (25 images)
Saved paired chunk 10/40 (25 images)
Saved paired chunk 11/40 (25 images)
Saved paired chunk 12/40 (25 images)
Saved paired chunk 13/40 (25 images)
Saved paired chunk 14/40 (25 images)
Saved paired chunk 15/40 (25 images)
Saved paired chunk 16/40 (25 images)
Saved paired chunk 17/40 (25 images)
Saved paired chunk 18/40 (25 images)
Saved paired chunk 19/40 (25 images)
Saved paired chunk 20/40 (25 images)
Saved paired chunk 21/40 (25 images)
Saved paired chunk 22/40 (25 images)
Saved paired chunk 23/40 (25 images)
Saved paired chunk 24/40 (25 images)
Saved paired chunk 25/40 (25 images)
Saved paired chunk 26/40 (25 images)
Saved paired chunk 27/40 (25 images)
Saved pair

In [8]:
from pathlib import Path
import subprocess

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

for title, args in [
    ("Working tree", ["status", "--short"]),
    ("Tracked changes", ["diff", "--stat"]),
    ("Staged changes", ["diff", "--cached", "--stat"]),
]:
    result = subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"\n--- {title} ---")
    print(result.stdout.strip() or "(none)")

print("\n--- Colab notebook candidates ---")
notebook_dir = Path("/content/drive/MyDrive/Colab Notebooks")
candidates = sorted(notebook_dir.glob("05a*.ipynb"))
for path in candidates:
    print(f"{path} | bytes={path.stat().st_size}")
if not candidates:
    print("No matching notebook found.")

print("\nRead-only inspection. No staging, commit or push performed.")


--- Working tree ---
?? docs/research_reports/05a_runner_notes_v1.md
?? docs/research_reports/05a_shared_INT8_clean_baseline_card_v1.md
?? src/baseline05a_runner_v1.py
?? tests/

--- Tracked changes ---
(none)

--- Staged changes ---
(none)

--- Colab notebook candidates ---
/content/drive/MyDrive/Colab Notebooks/05a_shared_clean_baseline.ipynb | bytes=28183

Read-only inspection. No staging, commit or push performed.


In [9]:
from pathlib import Path
import gzip
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
source = (
    repo / "results/notebook05a/shared_clean_cpu_v1/"
    "evaluation_and_quality_gate.json.gz"
)
destination = (
    repo / "docs/research_reports/"
    "05a_shared_INT8_clean_evaluation_v1.json"
)

expected_sha = (
    "8bb5cca90038987cf54457b7197036439d"
    "136167b71f65c49423454f7c37290b"
)

source_bytes = source.read_bytes()
source_sha = hashlib.sha256(source_bytes).hexdigest()
if source_sha != expected_sha:
    raise RuntimeError("Evaluation archive SHA256 mismatch.")

envelope = json.loads(gzip.decompress(source_bytes))
publication = {
    "source_relative_path": source.relative_to(repo).as_posix(),
    "source_archive_sha256": source_sha,
    "evaluation_record": envelope,
}
content = (
    json.dumps(
        publication,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ) + "\n"
).encode("utf-8")

if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Destination exists with different content.")
    print("Existing publication copy matches: PASS")
else:
    with destination.open("xb") as handle:
        handle.write(content)
    print("Saved:", destination)

if destination.read_bytes() != content:
    raise RuntimeError("Publication copy verification failed.")

print("Report SHA256:", hashlib.sha256(content).hexdigest())
print("Source archive unchanged: PASS")
print("No inference, staging, commit or push performed.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/05a_shared_INT8_clean_evaluation_v1.json
Report SHA256: 47861220401cc36ea2c6c19f427d351dcbb5ed39a9067fa79e0abbb175f889d7
Source archive unchanged: PASS
No inference, staging, commit or push performed.
